## Cell 1 – Imports and project paths

In [17]:

import numpy as np
import pandas as pd
import joblib
import mlflow

from pathlib import Path

from surprise import SVD, Dataset, Reader
from surprise.model_selection import GridSearchCV

# Project paths
PROCESSED_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Processed directory:", PROCESSED_DIR.resolve())
print("Model directory:", MODEL_DIR.resolve())
print("MLflow version:", mlflow.__version__)


Processed directory: C:\Users\Dell\Movie-recommendation-system\Backend\data\processed
Model directory: C:\Users\Dell\Movie-recommendation-system\Backend\models
MLflow version: 3.17.2


## Cell 2 – Load datasets and previous evaluation results

In [18]:

# Load cleaned ratings
ratings = pd.read_csv(
    PROCESSED_DIR / "ratings_clean.csv"
)

# Load Week 5 evaluation results
regression_df = pd.read_csv(
    PROCESSED_DIR / "regression_metrics.csv"
)

classification_df = pd.read_csv(
    PROCESSED_DIR / "classification_metrics.csv"
)

print("Ratings shape:", ratings.shape)

print("\nRegression Metrics:")
display(regression_df.round(4))

print("\nClassification Metrics:")
display(classification_df.round(4))


Ratings shape: (101696, 7)

Regression Metrics:


,Model,MAE,RMSE
0,Baseline (Mean),0.8353,1.0391
1,SVD,0.6676,0.8660
2,NCF,0.6650,0.8710



Classification Metrics:


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Baseline (Mean),0.4901,0.0000,0.0000,0.0000,0.5000
1,SVD,0.6416,0.8297,0.3740,0.5156,0.7829
2,NCF,0.6705,0.8090,0.4632,0.5891,0.7809


## Cell 3 – Prepare Surprise Dataset

In [19]:

# Define rating scale
reader = Reader(rating_scale=(0.5, 5))

# Prepare data for Surprise
data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

print("Surprise dataset prepared successfully!")


Surprise dataset prepared successfully!


## GridSearchCV

## Cell 4 – Hyperparameter tuning with GridSearchCV

In [20]:

# Define hyperparameter search space
param_grid = {
    "n_factors": [50, 100],
    "n_epochs": [20, 30],
    "lr_all": [0.002, 0.005],
    "reg_all": [0.02, 0.05],
    "random_state": [42]
}

# Initialize GridSearchCV
gs = GridSearchCV(
    SVD,
    param_grid,
    measures=["rmse", "mae"],
    cv=3,
    joblib_verbose=1
)

# Perform hyperparameter optimization
gs.fit(data)

# Display results
print("Best RMSE:", gs.best_score["rmse"])
print("Best MAE:", gs.best_score["mae"])

print("\nBest Parameters:")
print(gs.best_params["rmse"])


Best RMSE: 0.8623677612498463
Best MAE: 0.6640322793664364

Best Parameters:
{'n_factors': 50, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.05, 'random_state': 42}


[Parallel(n_jobs=1)]: Done  48 out of  48 | elapsed:  1.7min finished


## Cell 5 – Train the best SVD model

In [21]:

# Get best parameters based on RMSE
best_params = gs.best_params["rmse"]

print("Best parameters:", best_params)

# Create final SVD model
best_svd = SVD(**best_params)

# Build full training set
trainset = data.build_full_trainset()

# Train final model
best_svd.fit(trainset)

print("Best SVD model trained successfully!")


Best parameters: {'n_factors': 50, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.05, 'random_state': 42}
Best SVD model trained successfully!


## Cell 6 – Model serialization

In [22]:
# Define model path
svd_model_path = MODEL_DIR / "best_svd_model.pkl"

# Save trained model
joblib.dump(best_svd, svd_model_path)

print("Model saved successfully!")
print("Saved at:", svd_model_path.resolve())


Model saved successfully!
Saved at: C:\Users\Dell\Movie-recommendation-system\Backend\models\best_svd_model.pkl


## Cell 7 – Load and test the serialized model

In [23]:

# Load saved model
loaded_svd = joblib.load(svd_model_path)

print("Model loaded successfully!")

# Test prediction using an existing rating row
# Serialization verification (sample prediction)
sample = ratings.iloc[0]

test_prediction = loaded_svd.predict(
    uid=sample["userId"],
    iid=sample["movieId"]
)

print("Serialization test: SUCCESS")
print("Actual rating:", sample["rating"])
print("Predicted rating:", round(test_prediction.est, 4))

test_prediction = loaded_svd.predict(
    uid=sample["userId"],
    iid=sample["movieId"]
)

print("User ID:", sample["userId"])
print("Movie ID:", sample["movieId"])
print("Actual rating:", sample["rating"])
print("Predicted rating:", round(test_prediction.est, 4))


Model loaded successfully!
Serialization test: SUCCESS
Actual rating: 4.0
Predicted rating: 4.0591
User ID: 1
Movie ID: 16
Actual rating: 4.0
Predicted rating: 4.0591


## ML flow

## Cell 8 – Configure MLflow

In [24]:

# Create or select MLflow experiment
mlflow.set_experiment("Movie_Recommendation_System")

print("Experiment configured successfully!")
print("Tracking URI:", mlflow.get_tracking_uri())


Experiment configured successfully!
Tracking URI: sqlite:///c:/Users/Dell/Movie-recommendation-system/Backend/notebooks/mlflow.db


## Cell 9 – Track tuned SVD experiment

In [25]:

with mlflow.start_run(run_name="SVD_Tuned"):

    # Log model parameters
    mlflow.log_params(best_params)

    mlflow.log_param("model", "SVD")
    mlflow.log_param("cv_folds", 3)
    mlflow.log_param("optimization_metric", "RMSE")

    # Log cross-validation metrics
    mlflow.log_metric(
        "validation_rmse",
        float(gs.best_score["rmse"])
    )

    mlflow.log_metric(
        "validation_mae",
        float(gs.best_score["mae"])
    )

    # Log serialized model as an artifact
    mlflow.log_artifact(
        str(svd_model_path),
        artifact_path="models"
    )

    # Project information
    mlflow.set_tag(
        "project",
        "Movie Recommendation System"
    )

    mlflow.set_tag(
        "model_type",
        "Collaborative Filtering"
    )

    print("Tuned SVD experiment tracked successfully!")


Tuned SVD experiment tracked successfully!


## Cell 10 – Track previous model evaluation results

In [26]:

# Combine regression and classification metrics
evaluation_df = regression_df.merge(
    classification_df,
    on="Model",
    how="left"
)

# Track each model
for _, row in evaluation_df.iterrows():

    model_name = row["Model"]

    with mlflow.start_run(
        run_name=f"{model_name}_Evaluation"
    ):

        mlflow.log_param("model", model_name)

        # Log regression and classification metrics
        for metric in [
            "MAE",
            "RMSE",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "ROC-AUC"
        ]:

            if metric in row.index and pd.notna(row[metric]):

                mlflow.log_metric(
                    metric,
                    float(row[metric])
                )

        mlflow.set_tag(
            "project",
            "Movie Recommendation System"
        )

print("Previous model evaluation results tracked!")


Previous model evaluation results tracked!


## Cell 11 – Check MLflow experiments

In [27]:

print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())

experiment = mlflow.get_experiment_by_name(
    "Movie_Recommendation_System"
)

if experiment is None:

    print("Experiment not found!")

else:

    print("Experiment name:", experiment.name)
    print("Experiment ID:", experiment.experiment_id)

    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id]
    )

    print("\nNumber of runs:", len(runs))

    if not runs.empty:

        display(
            runs[[
                "tags.mlflow.runName",
                "status",
                "params.model",
                "metrics.MAE",
                "metrics.RMSE",
                "metrics.validation_mae",
                "metrics.validation_rmse"
            ]].round(4)
        )


MLflow version: 3.17.2
Tracking URI: sqlite:///c:/Users/Dell/Movie-recommendation-system/Backend/notebooks/mlflow.db
Experiment name: Movie_Recommendation_System
Experiment ID: 2

Number of runs: 34


,tags.mlflow.runName,status,params.model,metrics.MAE,metrics.RMSE,metrics.validation_mae,metrics.validation_rmse
0,NCF_Evaluation,FINISHED,NCF,0.6650,0.8710,NaN,NaN
1,SVD_Evaluation,FINISHED,SVD,0.6676,0.8660,NaN,NaN
2,Baseline (Mean)_Evaluation,FINISHED,Baseline (Mean),0.8353,1.0391,NaN,NaN
3,SVD_Tuned,FINISHED,SVD,NaN,NaN,0.6640,0.8624
4,NCF_Evaluation,FINISHED,NCF,0.6650,0.8710,NaN,NaN
5,SVD_Evaluation,FINISHED,SVD,0.6676,0.8660,NaN,NaN
6,Baseline (Mean)_Evaluation,FINISHED,Baseline (Mean),0.8353,1.0391,NaN,NaN
7,SVD_Tuned,FINISHED,SVD,NaN,NaN,0.6639,0.8616
8,NCF_Classification,FINISHED,NCF,NaN,NaN,NaN,NaN
9,SVD_Classification,FINISHED,SVD,NaN,NaN,NaN,NaN


## Cell 12 – Compare model evaluation results

In [28]:

# Select evaluation metrics
comparison = evaluation_df[
    [
        "Model",
        "MAE",
        "RMSE",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ]
].copy()

display(comparison.round(4))


,Model,MAE,RMSE,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Baseline (Mean),0.8353,1.0391,0.4901,0.0000,0.0000,0.0000,0.5000
1,SVD,0.6676,0.8660,0.6416,0.8297,0.3740,0.5156,0.7829
2,NCF,0.6650,0.8710,0.6705,0.8090,0.4632,0.5891,0.7809


## Cell 13 – Display tuned SVD results

In [29]:

tuning_results = pd.DataFrame([
    {
        "Model": "SVD Tuned",
        "Validation RMSE": gs.best_score["rmse"],
        "Validation MAE": gs.best_score["mae"],
        "Best Parameters": str(best_params)
    }
])

display(tuning_results.round(4))


,Model,Validation RMSE,Validation MAE,Best Parameters
0,SVD Tuned,0.8624,0.664,"{'n_factors': 50, 'n_epochs': 30, 'lr_all': 0...."


## Cell 14 – Final verification

In [30]:

print("=" * 50)
print("WEEK 6 - TUNING AND TRACKING SUMMARY")
print("=" * 50)

print("1. Hyperparameter tuning: COMPLETED")
print("2. Best SVD training: COMPLETED")
print("3. Model serialization: COMPLETED")
print("4. Model loading: COMPLETED")
print("5. MLflow tracking: COMPLETED")

print("\nBest validation RMSE:",
      round(gs.best_score["rmse"], 4))

print("Best validation MAE:",
      round(gs.best_score["mae"], 4))

print("\nModel file exists:",
      svd_model_path.exists())

print("\nTotal MLflow runs:",
      len(runs))

print("\nWeek 6 completed!")


WEEK 6 - TUNING AND TRACKING SUMMARY
1. Hyperparameter tuning: COMPLETED
2. Best SVD training: COMPLETED
3. Model serialization: COMPLETED
4. Model loading: COMPLETED
5. MLflow tracking: COMPLETED

Best validation RMSE: 0.8624
Best validation MAE: 0.664

Model file exists: True

Total MLflow runs: 34

Week 6 completed!
